# ETL Extract Notebook

This notebook is the first step of the ETL workflow. It connects to the local FastAPI service, pulls the business data from all available endpoints, and converts the raw API responses into clean staging datasets that can be used by downstream transformation and loading steps.

## What this notebook does
- Verifies that the FastAPI app is reachable.
- Seeds dummy data when needed.
- Extracts customer, product, order, feedback, sentiment, and analytics data.
- Converts the response payloads into pandas DataFrames.
- Saves the extracted data as CSV files in the ETL staging folder.

In [ ]:
# Cell 1 — Setup and configuration
from pathlib import Path
import pandas as pd
import requests

BASE_URL = "http://127.0.0.1:8000"
STAGING_DIR = Path("ETL/staging")
STAGING_DIR.mkdir(parents=True, exist_ok=True)

# Helper function for safe API calls

def fetch_json(path: str) -> dict:
    response = requests.get(f"{BASE_URL}{path}", timeout=30)
    response.raise_for_status()
    return response.json()

# Quick health check to confirm the API is available
health_response = requests.get(f"{BASE_URL}/health", timeout=20)
health_response.raise_for_status()
print("API health status:", health_response.json())

In [ ]:
# Cell 2 — Extract data from all API endpoints
endpoints = {
    "customers": "/customers",
    "products": "/products",
    "orders": "/orders",
    "feedback": "/feedback",
    "sentiments": "/sentiments",
    "feedback_summary": "/analytics/feedback-summary",
}

# Pull data from each endpoint and store the raw payloads in a dictionary
raw_data = {name: fetch_json(path) for name, path in endpoints.items()}

# Print a compact summary of what was retrieved
print("Retrieved datasets:", ", ".join(raw_data.keys()))
print("Customers count:", len(raw_data["customers"]))
print("Products count:", len(raw_data["products"]))
print("Orders count:", len(raw_data["orders"]))
print("Feedback count:", len(raw_data["feedback"]))
print("Sentiments count:", len(raw_data["sentiments"]))
print("Feedback summary:", raw_data["feedback_summary"])

In [ ]:
# Cell 3 — Convert raw JSON into DataFrames and save staging files
customers_df = pd.DataFrame(raw_data["customers"])
products_df = pd.DataFrame(raw_data["products"])
orders_df = pd.DataFrame(raw_data["orders"])
feedback_df = pd.DataFrame(raw_data["feedback"])
sentiments_df = pd.DataFrame(raw_data["sentiments"])
summary_df = pd.DataFrame([raw_data["feedback_summary"]])

# Normalize timestamp columns for downstream transformation
orders_df["order_date"] = pd.to_datetime(orders_df["order_date"])
feedback_df["feedback_date"] = pd.to_datetime(feedback_df["feedback_date"])

# Save each dataset as a CSV file for the next stage of the pipeline
customers_df.to_csv(STAGING_DIR / "customers.csv", index=False)
products_df.to_csv(STAGING_DIR / "products.csv", index=False)
orders_df.to_csv(STAGING_DIR / "orders.csv", index=False)
feedback_df.to_csv(STAGING_DIR / "feedback.csv", index=False)
sentiments_df.to_csv(STAGING_DIR / "sentiments.csv", index=False)
summary_df.to_csv(STAGING_DIR / "feedback_summary.csv", index=False)

print("Staging files written to:", STAGING_DIR)
print("\nCustomers preview")
customers_df.head()